# Tarea 2

**INF-396 Introducción a la Ciencia de Datos** · Universidad Técnica Federico Santa María

| | |
|---|---|
| **Integrante 1** | *(nombre)* |
| **Integrante 2** | *(nombre)* |

**Entrega**

1. Este notebook, con el nombre `tarea2_apellido1_apellido2.ipynb`, ejecutado de
   principio a fin en el entorno del curso (`uv sync`), leyendo los datos desde
   `datos/tarea2/`.
2. La parte 1 en un PDF escrito en LaTeX, `tarea2_apellido1_apellido2_parte1.pdf`.
3. Las interpretaciones van en las celdas de markdown marcadas como
   **Interpretación**. Un resultado correcto sin interpretación no otorga el puntaje
   completo.
4. Al final, la declaración de uso de IA generativa. Los contenidos de esta tarea
   entran en el Certamen 2: el uso de IA generativa se limita a consultas breves, y
   el código y las interpretaciones deben ser trabajo propio de la pareja.

## 0. Configuración

Esta celda lee los dos archivos y separa entrenamiento y prueba: 30% en prueba con la
semilla 396; en los tumores, de forma estratificada, sorteando el 30% de cada clase.
**Ejecútenla antes que todo lo demás y no cambien la semilla.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

RUTA = "../../datos/tarea2/"
SEMILLA = 396

# Viviendas de California: 8 variables y el precio mediano (cientos de miles de dólares)
viviendas = pd.read_csv(RUTA + "viviendas_california.csv")
X_viv = viviendas.drop(columns="precio_mediano")
y_viv = viviendas["precio_mediano"]
Xv_ent, Xv_pru, yv_ent, yv_pru = train_test_split(X_viv, y_viv, test_size=0.3, random_state=SEMILLA)

# Tumores de Wisconsin: 30 variables y el diagnóstico; la respuesta vale 1 si es maligno
tumores = pd.read_csv(RUTA + "tumores_wisconsin.csv")
X_tum = tumores.drop(columns="diagnostico")
y_tum = (tumores["diagnostico"] == "maligno").astype(int)
Xt_ent, Xt_pru, yt_ent, yt_pru = train_test_split(X_tum, y_tum, test_size=0.3, random_state=SEMILLA,
                                                  stratify=y_tum)

print("viviendas: entrenamiento", len(Xv_ent), "| prueba", len(Xv_pru))
print("tumores:   entrenamiento", len(Xt_ent), "| prueba", len(Xt_pru),
      "| malignos en entrenamiento", int(yt_ent.sum()), "| en prueba", int(yt_pru.sum()))

## Parte 1. Teoría (20 puntos)

Se entrega en el PDF `tarea2_apellido1_apellido2_parte1.pdf`, con las dos
demostraciones del enunciado.

## Parte 2. Implementación (25 puntos)

Viviendas de California: se predice el precio mediano de la zona a partir de las otras
8 variables. Estandaricen las 8 variables con el entrenamiento, centren el precio
restándole su media de entrenamiento (también en la prueba) y no usen intercepto, de
modo que la fórmula de Ridge aplique directamente.

### 3. Ridge con la fórmula (12 pts)

**Instrucciones**

La solución de Ridge es $b = (X^\top X + \lambda I)^{-1} X^\top y$. Programen esa fórmula
con numpy: una función que reciba la tabla estandarizada, la respuesta centrada y λ, y
devuelva los 8 coeficientes.

Para λ igual a 0, 10, 1.000 y 100.000, comparen sus coeficientes con los de
scikit-learn (`Ridge` sin intercepto; para λ igual a 0, `LinearRegression` sin
intercepto) y reporten la mayor diferencia en valor absoluto.

Grafiquen los 8 coeficientes contra λ, con λ en escala logarítmica.

En la interpretación, comenten qué pasa con los coeficientes al crecer λ.

In [ ]:
# Su código (agreguen las celdas que necesiten)

**Interpretación:** *(reemplacen este texto por su respuesta)*

### 4. Validación cruzada a mano (13 pts)

**Instrucciones**

Programen la validación cruzada de 5 grupos sin usar las funciones de validación
cruzada de scikit-learn: barajen los índices del entrenamiento con la semilla 396,
repártanlos en 5 grupos y, para cada λ de una grilla logarítmica entre 0,01 y 100.000,
ajusten con cuatro grupos usando su función del ítem 3 y midan el RMSE en el quinto,
rotando el grupo que valida. Grafiquen el error promedio contra λ y reporten el λ
elegido.

Comparen ese λ con el que elige `RidgeCV` con la misma grilla y cinco grupos.

En la interpretación, expliquen por qué los dos λ pueden no coincidir exactamente y
qué indica la forma de la curva sobre la necesidad de regularizar en este caso.

In [ ]:
# Su código (agreguen las celdas que necesiten)

**Interpretación:** *(reemplacen este texto por su respuesta)*

## Parte 3. Clasificación con costos (30 puntos)

Tumores de Wisconsin: la respuesta vale 1 si el tumor es maligno y 0 si es benigno.
Estandaricen las 30 variables con el entrenamiento.

### 5. Dos regresiones logísticas (8 pts)

**Instrucciones**

Sea $y_i \in \{0, 1\}$ el diagnóstico del tumor $i$ y $p_i = 1 / (1 + e^{-(a + x_i^\top b)})$
la probabilidad de maligno que le asigna el modelo, con $x_i \in \mathbb{R}^{30}$. La
regresión logística sin regularización elige $a$ y $b$ minimizando la pérdida logística
$L(a, b) = -\sum_i [\, y_i \log p_i + (1 - y_i) \log (1 - p_i) \,]$, que equivale a
maximizar la verosimilitud. La regresión logística con la penalización de Ridge
minimiza $L(a, b) + \lambda \|b\|^2$, sin penalizar el intercepto. En scikit-learn la
fuerza de la penalización se indica con $C = 1/\lambda$, y la logística viene
penalizada por defecto.

Ajusten ambos modelos en scikit-learn, no en statsmodels, con las 30 variables
estandarizadas; para el segundo, elijan C por validación cruzada de 5 grupos en el
entrenamiento. Reporten el mayor $|b_j|$ y el AUC en entrenamiento y en
prueba de cada modelo.

En la interpretación, expliquen la diferencia entre los AUC de entrenamiento y de
prueba de cada modelo.

In [ ]:
# Su código (agreguen las celdas que necesiten)

**Interpretación:** *(reemplacen este texto por su respuesta)*

### 6. Clasificar con umbral 0,5 (6 pts)

**Instrucciones**

Cada modelo asigna a cada tumor una probabilidad de ser maligno; clasifiquen como
maligno los tumores con probabilidad mayor o igual a 0,5. Para los dos modelos,
reporten en prueba la matriz de confusión, la sensibilidad, la especificidad y la
exactitud.

En la interpretación, indiquen cuál de los dos errores es más grave en este problema.

In [ ]:
# Su código (agreguen las celdas que necesiten)

**Interpretación:** *(reemplacen este texto por su respuesta)*

### 7. El costo de cada error (10 pts)

**Instrucciones**

Dejar pasar un tumor maligno cuesta 20 veces lo que cuesta una falsa alarma. Obtengan
el umbral que minimiza el costo esperado y aplíquenlo a los dos modelos. Reporten en
prueba la matriz de confusión y el costo total, 20·FN + FP, comparado con el del umbral
0,5.

Después fijen una meta, detectar al 99% de los malignos. Elijan el umbral que la cumple
usando solo el entrenamiento y reporten en prueba la sensibilidad y la especificidad
que logra.

En la interpretación, expliquen por qué un modelo responde al cambio de umbral y el
otro casi no.

In [ ]:
# Su código (agreguen las celdas que necesiten)

**Interpretación:** *(reemplacen este texto por su respuesta)*

### 8. Curva ROC y AUC (6 pts)

**Instrucciones**

Dibujen en una sola figura las curvas ROC en prueba de los dos modelos, marcando en
cada una los tres umbrales usados: 0,5, el de costos y el de la meta.

En la interpretación, con el AUC y las curvas, digan cuál modelo es mejor y si la
respuesta depende del umbral.

In [ ]:
# Su código (agreguen las celdas que necesiten)

**Interpretación:** *(reemplacen este texto por su respuesta)*

## Parte 4. Regresión con muchas variables (25 puntos)

Viviendas de California, ahora con intercepto y con la respuesta sin centrar: el precio
mediano de la zona, en cientos de miles de dólares. Aquí pueden elegir λ con las
funciones de scikit-learn.

### 9. Fabricar variables (5 pts)

**Instrucciones**

A partir de las 8 variables estandarizadas, construyan los 8 cuadrados y los 28
productos de pares: 44 variables en total. Vuelvan a estandarizarlas con el
entrenamiento.

Reporten las dimensiones de las tablas de entrenamiento y de prueba.

In [ ]:
# Su código (agreguen las celdas que necesiten)

**Dimensiones:** *(reemplacen este texto por su respuesta)*

### 10. Mínimos cuadrados, Ridge y Lasso (12 pts)

**Instrucciones**

Ajusten mínimos cuadrados con las 8 variables originales y con las 44, y Ridge y Lasso
con las 44, con λ elegido por validación cruzada de 5 grupos en el entrenamiento.

Reporten en una tabla el RMSE en entrenamiento y en prueba de los cuatro modelos, más
el de predecir siempre el precio promedio del entrenamiento. Grafiquen, para Ridge y
para Lasso, el RMSE de validación cruzada contra λ, marcando el λ elegido.

En la interpretación, expliquen qué le pasa a mínimos cuadrados al pasar de 8 a 44
variables, en el entrenamiento y en la prueba, y qué hacen Ridge y Lasso con eso.

In [ ]:
# Su código (agreguen las celdas que necesiten)

**Interpretación:** *(reemplacen este texto por su respuesta)*

### 11. Qué conserva Lasso (8 pts)

**Instrucciones**

Grafiquen los coeficientes de Lasso contra λ (los 44, o los más relevantes), con λ en
escala logarítmica, marcando el λ elegido. Digan cuántas de las 44 variables conserva Lasso con ese λ y cuáles son
las cinco de mayor coeficiente en valor absoluto.

Cierren con media página: cuál de los tres modelos usarían para tasar viviendas y por
qué, y si los coeficientes de ese modelo se pueden leer como efectos o solo sirven
para predecir.

In [ ]:
# Su código (agreguen las celdas que necesiten)

**Cierre:** *(reemplacen este texto por su respuesta)*

## Declaración de uso de IA generativa

*(Qué herramienta usaron y para qué, en una línea. Si no usaron ninguna, indíquenlo.)*